In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [19]:
# 1. INIT SPARK
spark = SparkSession.builder \
    .appName("Ad_Spend_Pipeline") \
    .config("spark.jars", "/Users/as-mac-1261/Downloads/mysql-connector-j-9.6.0/mysql-connector-j-9.6.0.jar") \
    .getOrCreate()

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


26/03/20 14:26:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [22]:

# 2. READ DATA
campaign_df = spark.read.csv("/Users/as-mac-1261/sample/sample/data/ads_camp.csv", header=True, inferSchema=True)
print(df.show())

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|2026-03-18|  Google|    AC1001|     CMP100|      Summer Sale|         ACTIVE|      Billable|          IN|     mobile|     INR|     120000|  5400|        210|18450.75|  39200|2026-03-19 01:10:00|           B1|
|18/03/2026|facebook|    AC1002|     CMP200|Electronics Blast|         ACTIVE|      Billable|          IN|    desktop|     INR|      98000|  5100|         95| 2

In [25]:
# STANDARDIZE PLATFORM
campaign_df = campaign_df.withColumn(
    "platform_clean",
    when(lower(col("platform")).isin("fb", "facebook", "face book"), "facebook")
    .when(lower(col("platform")).isin("ig", "insta", "instagram"), "instagram")
    .otherwise(lower(col("platform")))
)

print(df.show())

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|2026-03-18|  Google|    AC1001|     CMP100|      Summer Sale|         ACTIVE|      Billable|          IN|     mobile|     INR|     120000|  5400|        210|18450.75|  39200|2026-03-19 01:10:00|           B1|
|18/03/2026|facebook|    AC1002|     CMP200|Electronics Blast|         ACTIVE|      Billable|          IN|    desktop|     INR|      98000|  5100|         95| 2

In [26]:
# HANDLE MIXED DATE FORMATS
campaign_df = campaign_df.withColumn(
    "event_date_std",
    coalesce(
        to_date(col("event_date"), "yyyy-MM-dd"),
        to_date(col("event_date"), "dd/MM/yyyy"),
        to_date(col("event_date"), "yyyy/MM/dd")
    )
)

print(df.show())

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+
|2026-03-18|  Google|    AC1001|     CMP100|      Summer Sale|         ACTIVE|      Billable|          IN|     mobile|     INR|     120000|  5400|        210|18450.75|  39200|2026-03-19 01:10:00|           B1|
|18/03/2026|facebook|    AC1002|     CMP200|Electronics Blast|         ACTIVE|      Billable|          IN|    desktop|     INR|      98000|  5100|         95| 2

In [27]:
# CONVERT TIMESTAMP (for dedup)
campaign_df = campaign_df.withColumn("snapshot_ts", to_timestamp(col("snapshot_ts")))

In [28]:
# FILTER ACTIVE + BILLABLE
campaign_df = campaign_df.filter(
    (col("campaign_status") == "ACTIVE") &
    (col("billing_status") == "Billable")
)

In [11]:
# 5. DEDUPLICATE
window = Window.partitionBy("campaign_id", "event_dt").orderBy(col("snapshot_ts").desc())
df = df.withColumn("rn", row_number().over(window))
df = df.filter(col("rn") == 1).drop("rn")
print(df.show())

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+------------+----------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|platform_std|  event_dt|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+------------+----------+
|2026-03-18|  Google|    AC1001|     CMP100|      Summer Sale|         ACTIVE|      Billable|          IN|     mobile|     INR|     120000|  5400|        210|18450.75|  39200|2026-03-19 02:05:00|           B2|      google|2026-03-18|
|2026-03-20|  Google|    AC1010|    CMP1000|         Auto Ads|  

In [29]:
# DEDUPLICATION
window_spec = Window.partitionBy("campaign_id", "event_date_std") \
                    .orderBy(col("snapshot_ts").desc())

campaign_df = campaign_df.withColumn("row_num", row_number().over(window_spec)) \
                         .filter(col("row_num") == 1) \
                         .drop("row_num")


In [33]:
# CLEAN NUMERIC VALUES
campaign_df = campaign_df.fillna({
    "clicks": 0,
    "impressions": 0,
    "conversions": 0,
    "ad_spend": 0.0,
    "revenue": 0.0
})

campaign_df.show()


+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|platform_clean|event_date_std|                 ctr|           conv_rate|     cost_per_conv|        roas_value|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+
|2026-03-18|  Google|    AC1001|     CMP

In [31]:
campaign_df = campaign_df.filter(col("ad_spend") >= 0)

In [34]:

# DERIVED METRICS (safe division)
campaign_df = campaign_df.withColumn(
    "ctr",
    when(col("impressions") != 0, col("clicks") / col("impressions")).otherwise(0)
).withColumn(
    "conv_rate",
    when(col("clicks") != 0, col("conversions") / col("clicks")).otherwise(0)
).withColumn(
    "cost_per_conv",
    when(col("conversions") != 0, col("ad_spend") / col("conversions")).otherwise(0)
).withColumn(
    "roas_value",
    when(col("ad_spend") != 0, col("revenue") / col("ad_spend")).otherwise(0)
)

campaign_df.show()

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|platform_clean|event_date_std|                 ctr|           conv_rate|     cost_per_conv|        roas_value|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+
|2026-03-18|  Google|    AC1001|     CMP

In [44]:
# FLAG ANOMALIES

campaign_df = campaign_df.withColumn(
    "leak_flag",
    when((col("roas_value") < 1.0) & (col("ad_spend") > 15000), 1).otherwise(0)
).withColumn(
    "high_ctr_flag",
    when(col("ctr") > 0.045, 1).otherwise(0)
).withColumn(
    "zero_conv_flag",
    when(col("conversions") < 50, 1).otherwise(0)
)

campaign_df = campaign_df.withColumn(
    "total_flags",
    col("leak_flag") + col("high_ctr_flag") + col("zero_conv_flag")
)

campaign_df.show()

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+---------+-------------+--------------+-----------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|platform_clean|event_date_std|                 ctr|           conv_rate|     cost_per_conv|        roas_value|leak_flag|high_ctr_flag|zero_conv_flag|total_flags|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+-------------------

In [45]:
# AGGREGATION
final_result = campaign_df.groupBy(
    "campaign_id", "platform_clean", "event_date_std"
).agg(
    sum("ad_spend").alias("total_spend"),
    sum("clicks").alias("total_clicks"),
    sum("impressions").alias("total_impressions"),
    sum("conversions").alias("total_conversions"),
    avg("ctr").alias("avg_ctr"),
    avg("conv_rate").alias("avg_conversion_rate"),
    avg("cost_per_conv").alias("avg_cost_per_conversion"),
    avg("roas_value").alias("avg_roas"),
    sum("total_flags").alias("anomaly_count"))

campaign_df.show()

+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+--------------------+--------------------+------------------+------------------+---------+-------------+--------------+-----------+
|event_date|platform|account_id|campaign_id|    campaign_name|campaign_status|billing_status|country_code|device_type|currency|impressions|clicks|conversions|ad_spend|revenue|        snapshot_ts|load_batch_id|platform_clean|event_date_std|                 ctr|           conv_rate|     cost_per_conv|        roas_value|leak_flag|high_ctr_flag|zero_conv_flag|total_flags|
+----------+--------+----------+-----------+-----------------+---------------+--------------+------------+-----------+--------+-----------+------+-----------+--------+-------+-------------------+-------------+--------------+--------------+-------------------

In [46]:
# 10. WRITE TO MYSQL
mysql_url = "jdbc:mysql://localhost:3306/assignment"

properties = {
    "user": "root",
    "password": "Jeevan@123",
    "driver": "com.mysql.cj.jdbc.Driver"
}
final_df.show()
final_df.write.jdbc(url=mysql_url, table="campaign_triage", mode="overwrite", properties=properties)
print("data base connected succesfully")

+-----------+------------+----------+-----------+------------+-----------------+-----------------+--------------------+--------------------+-------------------+------------------+-------------+
|campaign_id|platform_std|  event_dt|total_spend|total_clicks|total_impressions|total_conversions|                 ctr|     conversion_rate|cost_per_conversion|              roas|anomaly_count|
+-----------+------------+----------+-----------+------------+-----------------+-----------------+--------------------+--------------------+-------------------+------------------+-------------+
|     CMP100|      google|2026-03-18|   18450.75|        5400|           120000|              210|               0.045| 0.03888888888888889|  87.86071428571428|2.1245748817798735|            0|
|    CMP1000|      google|2026-03-20|    22000.0|        4000|            90000|              100|0.044444444444444446|               0.025|              220.0|1.3636363636363635|            0|
|    CMP1100|    facebook|2026